In [1]:
%pip install -qU langchain langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Groq API Key: ")
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)
print("Complaint Classifier ready!")

Groq API Key:  ········


Complaint Classifier ready!


In [3]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool
CATEGORIES = {
    "Payment": "Payment, billing, or transaction problems",
    "Delivery": "Shipping, delays, or tracking problems",
    "Product": "Damaged, defective, or incorrect products",
    "Refund": "Refund, cancellation, or money-back problems",
    "Other": "Complaints that do not fit other categories"
}
class Complaint(BaseModel):
    """Structured information about a customer complaint."""
    category: Literal[
        "Payment", "Delivery", "Product", "Refund", "Other"
    ] = Field(description="Complaint category")
    priority: Literal[
        "Low", "Medium", "High"
    ] = Field(description="Complaint priority")
    sentiment: Literal[
        "Positive", "Neutral", "Negative"
    ] = Field(description="Customer sentiment")
    reason: str = Field(
        description="Short reason for the classification"
    )
    recommended_action: str = Field(
        description="Suggested next step for customer support"
    )

In [4]:
import csv
from datetime import datetime
@tool
def get_category_info(category: str) -> str:
    """Get information about a complaint category.
    Args:category: Payment, Delivery, Product, Refund, or Other"""
    return CATEGORIES.get(category.strip().title(),"Unknown category")
@tool
def suggest_priority(category: str) -> str:
    """Suggest a default priority for a complaint category.
    Args:category: Complaint category"""
    priorities = {
        "Payment": "High",
        "Delivery": "Medium","Product": "High","Refund": "Medium","Other": "Low"
    }
    return priorities.get(category.strip().title(),"Unknown category"
    )
@tool
def log_complaint(
    complaint: str,category: str,priority: str,sentiment: str,reason: str,recommended_action: str
) -> str:
    """Save a classified complaint to complaint_logs.csv.
    Use this only when the user explicitly asks to log or save
    the complaint. Do not invent complaint details.
    """
    filename = "complaint_logs.csv"
    row = [
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),complaint,category,priority,sentiment,reason,recommended_action
    ]
    headers = [
        "timestamp","complaint","category","priority","sentiment","reason","recommended_action"
    ]
    try:
        file_exists = os.path.exists(filename)
        with open(filename, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            if not file_exists:
                writer.writerow(headers)
            writer.writerow(row)
        return "Complaint saved successfully."
    except Exception as e:
        return f"Could not save complaint: {e}"

In [5]:
from langchain.agents import create_agent
SYSTEM = """
ROLE: You are an AI Customer Complaint Classification Assistant.
TASK: Analyze customer complaints and identify:
- Category: Payment, Delivery, Product, Refund, or Other
- Priority: Low, Medium, or High
- Sentiment: Positive, Neutral, or Negative
- Reason: Why the complaint belongs to this category
- Recommended Action: What customer support should do next
RULES:
Use get_category_info to understand categories when needed.
Use suggest_priority to check default priority when needed.
Never invent information not present in the complaint.
Use log_complaint only when the user explicitly asks to save or log.
Ask for clarification if the complaint is unclear.
FORMAT:
Respond clearly and concisely.
Use JSON format for complaint classifications.
Be helpful, professional, and friendly.
"""
agent = create_agent(
    model=llm,
    tools=[
        get_category_info,
        suggest_priority,
        log_complaint
    ],
    system_prompt=SYSTEM
)
history = []
def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({
            "messages": history + [
                {"role": "user", "content": text}
            ]
        })
        history = result["messages"]
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I couldn't process that complaint right now."
print(chat("My payment was deducted twice from my account."))
print(chat("What category was my previous complaint?"))
print(chat("Please save my previous complaint in the complaint log."))

```json
{
  "Category": "Payment",
  "Priority": "High",
  "Sentiment": "Negative",
  "Reason": "The customer reports a duplicate deduction from their account, indicating a billing error that requires immediate attention.",
  "Recommended Action": "Apologize for the inconvenience, verify the duplicate transaction, issue a refund for the extra charge, and confirm the account balance with the customer."
}
```
```json
{
  "Category": "Payment"
}
```
Your complaint has been logged successfully. If you need anything else, just let me know!
